# Stored Procedures, Functions, and Triggers

Up until now, every time we wanted the database to do something, we had to send it a raw SQL string from Python. But what if you have a massive, 50-line SQL query that updates inventory, calculates taxes, and emails a receipt? Sending that massive query over the network every time someone buys a coffee is slow and inefficient.

Instead, we can save that logic directly **inside the database server**. 

* **Stored Procedures**: Saved scripts that perform complex business logic (inserts, updates, deletes).
* **Functions**: Saved calculations that return a specific value and can be used inside `SELECT` statements.
* **Triggers**: Hidden traps that automatically execute code when data is changed.

*(Note: Our lightweight SQLite sandbox has some limitations here. It fully supports Triggers, but it does not support Stored Procedures, and it handles Functions a bit differently. We will use standard SQL for Procedures, and a very cool Python trick for Functions!)*

---

# 1. Stored Procedures (Standard SQL)
A Stored Procedure is like a Python function, but written entirely in SQL and saved permanently in the database. 

Instead of an application sending 10 different SQL commands to the database, the application just sends one command: `CALL ProcessOrder(101);`. The database then executes the 10 saved steps internally at lightning speed.

**Standard Syntax (PostgreSQL / MySQL):**
```sql
-- Creating a Stored Procedure
CREATE PROCEDURE GiveCompanyRaise(IN raise_percentage DECIMAL)
BEGIN
    -- Update the salary of every active employee
    UPDATE Employees 
    SET salary = salary + (salary * raise_percentage)
    WHERE status = 'Active';
    
    -- Log the event in an audit table
    INSERT INTO Company_Events (event_name, date) 
    VALUES ('Company-wide Raise', CURRENT_TIMESTAMP);
END;

-- To execute this in the future, you simply run:
-- CALL GiveCompanyRaise(0.05);
```

*Why Data Scientists care:* If a Data Engineer builds a stored procedure to clean and format a daily data dump, you don't have to write the cleaning code yourself. You just `CALL CleanDailyData();` before running your models.

# 2. User-Defined Functions (UDFs) with Python & SQLite
A Function is similar to a Procedure, but it **must return a value**, and it is meant to be used *inside* a query, just like `SUM()` or `MAX()`.

While SQLite doesn't let you write custom functions in SQL, it allows something even better for Data Scientists: **You can write a function in Python, and inject it into the SQLite database to use in your SQL queries!**

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.executescript("""
CREATE TABLE Sales (id INTEGER, amount DECIMAL, state TEXT);
INSERT INTO Sales VALUES (1, 100.00, 'NY'), (2, 50.00, 'CA'), (3, 200.00, 'TX');
""")

# 1. Write a standard Python function
def calculate_regional_tax(amount, state):
    tax_rates = {'NY': 0.08, 'CA': 0.10, 'TX': 0.00}
    rate = tax_rates.get(state, 0.05) # Default to 5%
    return round(amount * rate, 2)

# 2. Register the Python function inside the SQLite database
# conn.create_function("SQL_NAME", number_of_arguments, python_function_name)
conn.create_function("CALC_TAX", 2, calculate_regional_tax)

# 3. Use your custom function inside a standard SQL query!
query_udf = """
SELECT 
    state, 
    amount, 
    CALC_TAX(amount, state) as tax_owed,
    amount + CALC_TAX(amount, state) as total_price
FROM Sales;
"""
print("--- Using a Custom Python Function inside SQL ---")
display(pd.read_sql_query(query_udf, conn))

--- Using a Custom Python Function inside SQL ---


,state,amount,tax_owed,total_price
0,NY,100,8.0,108.0
1,CA,50,5.0,55.0
2,TX,200,0.0,200.0


# 3. Triggers (Database Automation)
A Trigger is a piece of code that the database runs **automatically** when a specific event happens (`INSERT`, `UPDATE`, or `DELETE`).

Triggers are most commonly used for **Audit Logging**—keeping a strict history of who changed what, and when. Let's create an audit log that tracks every time someone's salary is changed.

In [2]:
# 1. Create our main table and an empty Audit Log table
cursor.executescript("""
CREATE TABLE Staff (emp_id INTEGER PRIMARY KEY, name TEXT, salary DECIMAL);

CREATE TABLE Salary_Audit (
    log_id INTEGER PRIMARY KEY,
    emp_id INTEGER,
    old_salary DECIMAL,
    new_salary DECIMAL,
    change_date DATETIME DEFAULT CURRENT_TIMESTAMP
);

INSERT INTO Staff (name, salary) VALUES ('Alice', 90000), ('Bob', 75000);
""")

# 2. Create the Trigger
# We tell it to watch the 'Staff' table. If an UPDATE happens on the 'salary' column, it fires.
# 'OLD' and 'NEW' are special keywords that hold the before and after data!
cursor.executescript("""
CREATE TRIGGER log_salary_changes
AFTER UPDATE OF salary ON Staff
BEGIN
    INSERT INTO Salary_Audit (emp_id, old_salary, new_salary)
    VALUES (OLD.emp_id, OLD.salary, NEW.salary);
END;
""")

# 3. Let's test it by giving Alice and Bob a raise
cursor.execute("UPDATE Staff SET salary = 95000 WHERE name = 'Alice';")
cursor.execute("UPDATE Staff SET salary = 80000 WHERE name = 'Bob';")

print("\n--- The Staff Table (After Raises) ---")
display(pd.read_sql_query("SELECT * FROM Staff;", conn))

print("\n--- The Automatically Generated Audit Log ---")
# We never wrote an INSERT statement for the Audit table in Python! The Trigger did it automatically.
display(pd.read_sql_query("SELECT * FROM Salary_Audit;", conn))

conn.close()


--- The Staff Table (After Raises) ---


,emp_id,name,salary
0,1,Alice,95000
1,2,Bob,80000



--- The Automatically Generated Audit Log ---


,log_id,emp_id,old_salary,new_salary,change_date
0,1,1,90000,95000,2026-04-10 16:44:19
1,2,2,75000,80000,2026-04-10 16:44:19


## Real-World Use Case or Analogy:
Think of these three concepts like running a **High-End Restaurant**:

* **Stored Procedure**: The Recipe Book. When a customer orders the "Chef's Special," the waiter doesn't shout step-by-step instructions to the kitchen (chop onions, sear meat, plate food). The waiter just shouts `CALL ChefsSpecial()`. The kitchen already has the complex, multi-step instructions saved and executes them perfectly.
* **Function**: A specialized kitchen appliance, like a blender. You pass ingredients (arguments) into the blender, press a button, and it returns a perfectly blended smoothie (a single value) that you can hand to the customer. 
* **Trigger**: The smoke detector. You don't actively run the smoke detector. It sits in the background, constantly watching. The moment it senses an `UPDATE` in the air quality (smoke), it automatically executes its emergency protocol (spraying water and logging the event to the fire department).

---